### Симуляции

In [105]:
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

mu, sigma = 40, 8
normal = np.random.normal(mu, sigma, size=5000)

fig = px.histogram(x=normal, histnorm='probability density', title="Normal distribution")

x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
y_theory = stats.norm.pdf(x_range, loc=mu, scale=sigma)

fig.add_trace(
     go.Scatter(
        x=x_range,
        y=y_theory,
        mode='lines',
        name='Theoretical',
        line={'color': 'red', 'width': 2}
    )
)

fig.show()

print(f"Sample mean: {normal.mean():.2f}, sample std: {normal.std(ddof=1):.2f}")
print(f"Theoretical mean: {mu:.2f}, theoretical std: {sigma:.2f}")

Sample mean: 39.94, sample std: 8.10
Theoretical mean: 40.00, theoretical std: 8.00


In [106]:
n, p = 1000, 0.04
binomial = np.random.binomial(n, p, size=5000)

fig = px.histogram(x=binomial, histnorm='probability density', title="Binomial distribution")

x_range = np.arange(binomial.min(), binomial.max() + 1)
y_theory = stats.binom.pmf(x_range, n, p)

fig.add_trace(
     go.Scatter(
        x=x_range,
        y=y_theory,
        mode='lines',
        name='Theoretical',
        line={'color': 'red', 'width': 2}
    )
)

fig.show()

print(f"Sample mean: {binomial.mean():.2f}, sample std: {binomial.std(ddof=1):.2f}")
print(f"Theoretical mean: {n*p:.2f}, theoretical std: {np.sqrt(n*p*(1-p)):.2f}")

Sample mean: 39.93, sample std: 6.14
Theoretical mean: 40.00, theoretical std: 6.20


In [53]:
sample_size = 500
experiments_cnt = 5000
beta = 100

means = [np.random.exponential(scale=beta, size=sample_size).mean() for _ in range(experiments_cnt)]

fig = px.histogram(x=means, histnorm='probability density', title="CLT")

mu = beta
sigma = beta/np.sqrt(sample_size)
x_range = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
y_theory = stats.norm.pdf(x_range, loc=mu, scale=sigma)

fig.add_trace(
     go.Scatter(
        x=x_range,
        y=y_theory,
        mode='lines',
        name='Theoretical',
        line={'color': 'red', 'width': 2}
    )
)

fig.show()


По центральной предельной теореме выборочное среднее значений со средним mu и со стандартным отклонением sigma имеет нормальное распределение со средним mu и со стандартным отклонением sigma / sqrt(n). Для экспоненциального распределения среднее и стандартное отклонение равны параметру beta (1/alpha). Соответственно получившееся распределение имеет среднее beta и стандартное отклонение beta / sqrt(n).

### Кейс про кнопку

SE = sqrt(p*(1-p)/n)

In [66]:
n = 5000
p1 = 0.031
p2 = 0.034
se1 = np.sqrt(p1*(1-p1)/n)
se2 = np.sqrt(p2*(1-p2)/n)
print(f"Стандартная ошибка для старой кнопки (p = {p1}): {100*se1:.3f} п.п.")
print(f"Стандартная ошибка для новой кнопки (p = {p2}): {100*se2:.3f} п.п.")

Стандартная ошибка для старой кнопки (p = 0.031): 0.245 п.п.
Стандартная ошибка для новой кнопки (p = 0.034): 0.256 п.п.


In [67]:
se_diff = np.sqrt(se1 ** 2 + se2 ** 2)
print(f"Стандартная ошибка разности конверсий: {100*se_diff:.3f} п.п.")

Стандартная ошибка разности конверсий: 0.355 п.п.


Вывод: стандартная ошибка разности конверсий больше получившейся реальной разницы. Значит изменение не отличить от шума. 

In [ ]:
sample_size = 5000
experiments_cnt = 5000
convertion = (155 + 170) / (2*sample_size)

print(f"Пусть конверсия одинаковая и равна {convertion:.2%}")
# print(f"Стандартное отклонение при n={sample_size*2} и p={convertion:.2f}: sqrt(np(1-p))={np.sqrt(2*sample_size*convertion*(1-convertion)):.2f}")

differences = []
for exp_ind in range(experiments_cnt):
    groupA = np.random.binomial(1, convertion, size=sample_size)
    groupB = np.random.binomial(1, convertion, size=sample_size)

    differences.append(groupA.mean() - groupB.mean())

differences = np.array(differences)
extreme = differences[abs(differences) > 0.003].size
print(f"Количество экстремальных значений (по модулю > 0.3 п.п.): {extreme} (p-value={extreme/sample_size:.3f})")

fig = px.histogram(x=differences, title="Differences")
fig.show()

Пусть конверсия одинаковая и равна 3.25%
Количество экстремальных значений (по модулю > 0.3 п.п.): 2028 (p-value=0.406)


Вывод: так как такая разница или более экстремальная возникает примерно в 40% случаев, почти наверное изменение конверсии это шум.

In [108]:
sessions = np.array([3, 4, 4, 5, 5, 6, 6, 7, 8, 52])
estimated_mean = sessions.mean()
estimated_std = sessions.std(ddof=1)
print("Выбросы по правилу трех сигм:", 
sessions[(sessions < estimated_mean - 3*estimated_std) | (sessions > estimated_mean + 3*estimated_std)])

p25, p75 = np.percentile(sessions, [25, 75])
print("Выбросы по персентилям:", 
sessions[(sessions < p25 - 1.5 * (p75 - p25)) | (sessions > p75 + 1.5 * (p75 - p25))])


Выбросы по правилу трех сигм: []
Выбросы по персентилям: [52]


Сам выброс сдвинул среднее вправо и увеличил дисперсию, поэтому правило 3 сигм не сработало. Персентили не учитывают насколько далеко оказались крайние выбросы, поэтому этот метод нашел сессию с нестандартным активным временем.